# 09 — Random Forest (sklearn + Optuna)

**Modelo de ensamble basado en bagging de árboles de decisión.**
El Random Forest selecciona un subconjunto aleatorio de features en cada split,
lo que reduce la correlación entre los árboles y mejora la generalización.

Referencia de clase: `modelos R/árboles/2-Pipeline_Forest_Clasificacion.R`
(grid search sobre `mtry` × `maxnodes`, evaluación por OOB error)

**Estrategia:**
- Los árboles manejan NaN nativamente → **sin imputación**
- `class_weight='balanced'` para el desbalance 8%/92%
- Optuna (20 trials): función objetivo = CV AUC − 0.5·(train AUC − CV AUC)
- OOB score como sanity check (análogo al OOB del R script)
- Métricas finales sobre train completo

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, confusion_matrix, f1_score, roc_curve
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

print('Imports OK')

In [ ]:
SEED     = 42
N_TRIALS = 20
np.random.seed(SEED)

BASE_DIR = Path('../../')
DATA_DIR = BASE_DIR / 'data/processed'

matplotlib.rcParams.update({'figure.figsize': (9, 6), 'font.size': 12,
                            'axes.spines.top': False, 'axes.spines.right': False})

In [ ]:
df      = pd.read_parquet(DATA_DIR / 'features_train.parquet')
df_test = pd.read_parquet(DATA_DIR / 'features_test.parquet')

feature_cols = [c for c in df.columns if c not in ('SK_ID_CURR', 'TARGET')]

X           = df[feature_cols].values
y           = df['TARGET'].values
X_test      = df_test[feature_cols].values
sk_ids_test = df_test['SK_ID_CURR'].values

# Nota: Random Forest maneja NaN en sklearn >= 1.4 con missing_values=np.nan
# Para versiones anteriores, imputar con mediana
import sklearn
sklearn_ver = tuple(int(x) for x in sklearn.__version__.split('.')[:2])
NATIVE_NAN  = sklearn_ver >= (1, 4)

if not NATIVE_NAN:
    print('sklearn < 1.4 detectado: imputando NaN con mediana para RF')
    from sklearn.impute import SimpleImputer
    imp = SimpleImputer(strategy='median')
    X   = imp.fit_transform(X)
    X_test = imp.transform(X_test)
else:
    print('sklearn >= 1.4: RF maneja NaN nativamente')

print(f'Train: {X.shape}  | Default rate: {y.mean():.2%}')
print(f'NaNs en X: {np.isnan(X).sum():,}')

In [ ]:
def compute_metrics(y_true, y_prob, threshold=0.5, label='Model'):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    auc  = roc_auc_score(y_true, y_prob)
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    return dict(Model=label, AUC=round(auc,4),
                N=len(y_true), P=int(y_pred.sum()),
                TP=int(tp), TN=int(tn), FP=int(fp), FN=int(fn),
                Recall=round(rec,4), Precision=round(prec,4), F1=round(f1,4))

def plot_roc_curve(y_true, y_prob, label, color, ax, title='ROC Curve'):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc = roc_auc_score(y_true, y_prob)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{label} (AUC = {auc:.4f})')
    ax.plot([0,1],[0,1],'k--',lw=1,alpha=0.5)
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title(title); ax.legend()
    return ax

print('Funciones definidas ✓')

## 1. Optuna — Búsqueda de Hiperparámetros

**Función objetivo** (anti-overfitting, equivalente al R file que usa OOB):
$$\text{objetivo} = \text{AUC}_{\text{OOB}} - 0.5 \cdot \max(0,\ \text{AUC}_{\text{train}} - \text{AUC}_{\text{OOB}})$$

El OOB (out-of-bag) es el estimador out-of-sample natural del Random Forest,
análogo al OOB error del R script.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

def objective_rf(trial):
    n_estimators      = trial.suggest_int('n_estimators',   100, 600)
    max_depth         = trial.suggest_int('max_depth',        4,  25, log=False)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
    min_samples_leaf  = trial.suggest_int('min_samples_leaf',  1, 10)
    max_features      = trial.suggest_categorical('max_features',
                                                   ['sqrt', 'log2', 0.3, 0.5])

    rf = RandomForestClassifier(
        n_estimators      = n_estimators,
        max_depth         = max_depth,
        min_samples_split = min_samples_split,
        min_samples_leaf  = min_samples_leaf,
        max_features      = max_features,
        class_weight      = 'balanced',
        oob_score         = True,
        n_jobs            = -1,
        random_state      = SEED
    )
    rf.fit(X, y)

    # OOB AUC
    oob_prob  = rf.oob_decision_function_[:, 1]
    oob_auc   = roc_auc_score(y, oob_prob)

    # Train AUC
    train_auc = roc_auc_score(y, rf.predict_proba(X)[:, 1])

    gap = max(0.0, train_auc - oob_auc)
    return oob_auc - 0.5 * gap

print(f'Lanzando Optuna — {N_TRIALS} trials (RF + OOB)...')
study_rf = optuna.create_study(
    direction = 'maximize',
    sampler   = optuna.samplers.TPESampler(seed=SEED)
)
study_rf.optimize(objective_rf, n_trials=N_TRIALS, show_progress_bar=True)

print(f'\nMejores hiperparámetros:')
for k, v in study_rf.best_params.items():
    print(f'  {k:<22}: {v}')
print(f'\nObjetivo (OOB AUC penalizado): {study_rf.best_value:.4f}')

In [ ]:
# Historial de Optuna
trial_df = study_rf.trials_dataframe()
param_cols = [c for c in trial_df.columns if c.startswith('params_')]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(trial_df['number'], trial_df['value'], 'o-', color='#2ecc71', ms=6)
ax.axhline(study_rf.best_value, color='#e74c3c', ls='--',
           label=f'Best = {study_rf.best_value:.4f}')
ax.set_xlabel('Trial'); ax.set_ylabel('Objetivo (OOB AUC penalizado)')
ax.set_title('Optuna — Random Forest')
ax.legend()
plt.tight_layout()
plt.show()


## 2. Modelo Final — Refit en Train Completo

In [ ]:
best_p = study_rf.best_params

rf_final = RandomForestClassifier(
    n_estimators      = best_p['n_estimators'],
    max_depth         = best_p['max_depth'],
    min_samples_split = best_p['min_samples_split'],
    min_samples_leaf  = best_p['min_samples_leaf'],
    max_features      = best_p['max_features'],
    class_weight      = 'balanced',
    oob_score         = True,
    n_jobs            = -1,
    random_state      = SEED
)

print('Entrenando modelo final...')
rf_final.fit(X, y)

oob_prob    = rf_final.oob_decision_function_[:, 1]
oob_auc     = roc_auc_score(y, oob_prob)
print(f'OOB AUC (sanity check): {oob_auc:.4f}')

## 3. Métricas sobre Train Completo

In [ ]:
y_prob_train = rf_final.predict_proba(X)[:, 1]
metrics = compute_metrics(y, y_prob_train, threshold=0.5, label='Random Forest')
metrics['OOB_AUC'] = round(oob_auc, 4)

print('=' * 60)
print('RANDOM FOREST — MÉTRICAS FINALES')
print('=' * 60)
for k, v in metrics.items():
    if k != 'Model':
        print(f'  {k:<12}: {v}')
print('=' * 60)

display(pd.DataFrame([metrics]).set_index('Model'))

## 4. Curva ROC

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ROC train
plot_roc_curve(y, y_prob_train, label='RF (train)', color='#e74c3c',
               ax=axes[0], title='ROC — Random Forest (train)')

# ROC OOB (out-of-bag — estimador out-of-sample)
plot_roc_curve(y, oob_prob, label='RF (OOB)', color='#3498db',
               ax=axes[1], title='ROC — Random Forest (OOB)')

plt.tight_layout()
plt.show()


## 5. Importancia de Variables (Top 15)

In [ ]:
importances = rf_final.feature_importances_
feat_imp_df = pd.DataFrame({'Feature': feature_cols, 'Importance': importances})
feat_imp_df = feat_imp_df.sort_values('Importance', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(9, 7))
bars = ax.barh(feat_imp_df['Feature'][::-1], feat_imp_df['Importance'][::-1],
               color='#2ecc71', edgecolor='white')
ax.set_xlabel('Importancia (Gini impurity decrease)')
ax.set_title('Top 15 Features — Random Forest')

# Etiquetas de valor
for bar, val in zip(bars, feat_imp_df['Importance'][::-1]):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()
display(feat_imp_df.reset_index(drop=True))

## 6. Distribución de Scores (Target 0 vs 1)

In [ ]:
from scipy.stats import gaussian_kde

fig, ax = plt.subplots(figsize=(9, 5))
for val, color, label in [(0, '#3498db', 'TARGET=0 (paga)'),
                           (1, '#e74c3c', 'TARGET=1 (default)')]:
    probs = y_prob_train[y == val]
    kde = gaussian_kde(probs, bw_method=0.1)
    xs  = np.linspace(0, 1, 300)
    ax.plot(xs, kde(xs), color=color, lw=2, label=label)
    ax.fill_between(xs, kde(xs), alpha=0.15, color=color)

ax.set_xlabel('Score predicho P(default)')
ax.set_ylabel('Densidad')
ax.set_title('Distribución de scores — Random Forest')
ax.legend()
plt.tight_layout()
plt.show()


## 7. Predicciones Test

In [ ]:
y_test_prob = rf_final.predict_proba(X_test)[:, 1]
submission  = pd.DataFrame({'SK_ID_CURR': sk_ids_test, 'TARGET': y_test_prob})
out_path = DATA_DIR / 'submission_09_rfr.csv'
submission.to_csv(out_path, index=False)

print(f'Predicciones guardadas: {out_path}')
print(f'Shape: {submission.shape}')
print(f'Score medio predicho: {y_test_prob.mean():.4f}')
display(submission.head())

## Resumen de Hiperparámetros Óptimos

In [ ]:
print('=' * 60)
print('RANDOM FOREST — HIPERPARÁMETROS ÓPTIMOS')
print('=' * 60)
for k, v in best_p.items():
    print(f'  {k:<22}: {v}')
print('\nMÉTRICAS FINALES')
print('-' * 60)
for k, v in metrics.items():
    if k != 'Model':
        print(f'  {k:<12}: {v}')
print('=' * 60)

## Kaggle Submission — AUC Test (Public Leaderboard)

Envía el CSV y recupera el AUC en el Public LB (~30% del test set).
Polling cada 30 s hasta que el status sea `complete` (máx 3 min).

> **Límite**: 5 submissions/día.


## Kaggle Submission — AUC Test (Public Leaderboard)

Envía el CSV a `home-credit-default-risk` y recupera el AUC del Public LB (~30% del test).
Usa la Python API (`KaggleApi`) directamente — polling cada 10 s, máx 5 min.

> **Límite**: 5 submissions/día.


In [ ]:
from kaggle import KaggleApi
import time

COMPETITION = 'home-credit-default-risk'
_sub_path   = DATA_DIR / 'submission_09_rfr.csv'
_msg = f"09_random_forest | train={metrics["AUC"]:.4f} | OOB={oob_auc:.4f}"

def _as_str(x):
    return '' if x is None else str(x)

def _get_score(api, comp, file_name, message, poll=10, timeout=300):
    """Poll until public_score appears on the matching submission."""
    start = time.time()
    while time.time() - start < timeout:
        subs = api.competition_submissions(comp)
        matched = next(
            (s for s in subs
             if _as_str(getattr(s, 'file_name', None)) == file_name
             and _as_str(getattr(s, 'description', None)) == message),
            subs[0] if subs else None
        )
        if matched:
            pub     = _as_str(getattr(matched, 'public_score',  None))
            status  = _as_str(getattr(matched, 'status',        None))
            elapsed = int(time.time() - start)
            print(f'  [{elapsed:>3}s] status={status!r}  public_score={pub!r}')
            if pub and pub.lower() not in ('', 'none', 'null', '-'):
                priv = _as_str(getattr(matched, 'private_score', None))
                return float(pub), (float(priv) if priv and priv.lower() not in ('','none','null','-') else None)
        time.sleep(poll)
    return None, None

_api = KaggleApi()
_api.authenticate()

print(f'Enviando: {_msg}')
_api.competition_submit(file_name=str(_sub_path), message=_msg, competition=COMPETITION)
print('Esperando scoring (poll 10 s / máx 5 min)...')

public_auc, private_auc = _get_score(_api, COMPETITION, _sub_path.name, _msg)
print(f'\nAUC test  Public  LB : {public_auc}')
print(f'AUC test  Private LB : {private_auc}')
